## Cleaning and transforming data and moving to silver layer ##

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name","ecommerce","catalog_name")
dbutils.widgets.text("storage_account_name","abiecommerceadlsdev","storage_account_name")
dbutils.widgets.text("container_name","ecomm-raw-data","container_name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

In [0]:
 silver_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoints/silver/          fact_order_items/"

In [0]:
df_bronze_orders = spark.readStream \
    .format("delta") \
    .table(f"{catalog_name}.bronze.brz_order_items")

        

In [0]:
df_silver_orders = df_bronze_orders.dropDuplicates(["order_id","item_seq"])
### converting text "two" in quantity column to number "2"
df_silver_orders = df_silver_orders.withColumn("quantity", F.when(F.col("quantity") == "Two", 2).otherwise(F.col("quantity")).cast("int"))
df_silver_orders = df_silver_orders.withColumn("unit_price", F.regexp_replace("unit_price", "[$]", "").cast("double"))
df_silver_orders = df_silver_orders.withColumn("discount_pct", F.regexp_replace("discount_pct", "%", "").cast("double"))
#df_silver_orders = df_silver_orders.withColumn("tax_amount", F.when(F.col("tax_amount") == "null","0").otherwise(F.col("tax_amount")).cast("double"))
df_silver_orders = df_silver_orders.withColumn("coupon_code", F.lower(F.trim(F.col("coupon_code"))))



In [0]:
df_silver_orders = df_silver_orders.withColumn("channel", F.when(F.col("channel") == "app","mobile").when(F.col("channel") == "web","Website").otherwise(F.col("channel"))).withColumn("processed_time", F.current_timestamp())


In [0]:
def upsert_to_silver(microBatchDF, batchId):
    table_name = f"{catalog_name}.silver.slv_order_items"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("silver_table").merge(
            microBatchDF.alias("batch_table"),
            "silver_table.order_id = batch_table.order_id AND silver_table.item_seq = batch_table.item_seq",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()    


In [0]:
# This line is running a Structured Streaming job that:
# - Reads incremental data from Bronze (df).
# - For each batch → applies upsert_to_silver (update if exists, insert if not).
# - Writes into a Silver Delta table with schema evolution enabled.
# - Uses checkpointing for recovery.
# - Runs in batch-like mode (once or availableNow), not continuous streaming.

df_silver_orders.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_silver
).format("delta").option("checkpointLocation", silver_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()